# AutoStock — Strategy Backtester

**Methodology**
1. Pick a *separation date*. Everything before → training data used to evaluate strategies. Everything after → simulation.
2. On the separation date, run all 5 strategies on every ticker's training slice.
3. For each triggered stock, enter at the **close price on the signal day**.
   - 1 strategy triggered → invest **\$1 000**
   - 2 strategies → **\$2 000**, and so on.
4. Hold until the **stop loss**, **trail/target exit**, or **252 trading-day max** is reached.
5. Repeat across 5 quarterly separation dates; aggregate results by strategy.

> **Note:** This is a simplified simulation (no slippage, no bid-ask spread, daily-bar resolution). Use as a relative ranking of strategies, not an exact profit estimate.

In [ ]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))

from strategy import STRATEGIES, check_all_strategies

DATA_DIR               = Path('data')
INVESTMENT_PER_STRAT   = 1_000   # $ per triggered strategy
MAX_HOLD_DAYS          = 252     # ≈ 1 trading year

# 5 quarterly separation dates — each needs ≥210 training bars
SEPARATION_DATES = [
    '2025-03-03',   # ~215 training bars, ~270 test bars
    '2025-06-02',   # ~275 training bars, ~210 test bars
    '2025-09-02',   # ~340 training bars, ~150 test bars
    '2025-12-01',   # ~400 training bars, ~ 90 test bars
    '2026-02-03',   # ~460 training bars, ~ 50 test bars
]

STRATEGY_NAMES = [name for name, _ in STRATEGIES]
print('Strategies:', STRATEGY_NAMES)
print('Separation dates:', SEPARATION_DATES)

In [ ]:
def load_all_stocks(min_rows: int = 210) -> dict:
    """Load every parquet in data/ into memory. Skip files with < min_rows bars."""
    stock_data = {}
    for p in sorted(DATA_DIR.glob('*.parquet')):
        ticker = p.stem
        try:
            df = pd.read_parquet(p)
            df.index = pd.to_datetime(df.index, utc=True)
            if len(df) >= min_rows:
                stock_data[ticker] = df
        except Exception:
            pass
    return stock_data

stock_data = load_all_stocks()
all_dates  = [d for df in stock_data.values() for d in [df.index.min(), df.index.max()]]
print(f'Loaded {len(stock_data)} tickers  '
      f'({sum(len(v) for v in stock_data.values()):,} total bars)')
print(f'Date range: {min(all_dates).date()} → {max(all_dates).date()}')

In [ ]:
# ── Stop price (hard cut-loss level) ─────────────────────────────────────────

def compute_stop(strategy_name: str, entry_price: float, row: pd.Series) -> float:
    """Compute the hard stop price matching each strategy's own logic."""
    atr   = row.get('ATR14',    np.nan)
    ma20  = row.get('MA20',     np.nan)
    ma200 = row.get('MA200',    np.nan)
    bb_lo = row.get('BB_lower', np.nan)

    if strategy_name in ('MA Pullback', 'Trend+Pullback+Momentum'):
        return (entry_price - 1.5 * atr) if pd.notna(atr) else entry_price * 0.92

    if strategy_name == 'Bollinger+RSI MeanReversion':
        if pd.notna(bb_lo) and pd.notna(atr):
            return bb_lo - atr
        return entry_price * 0.92

    if strategy_name == 'Golden/Death Cross':
        return (ma200 * 0.98) if pd.notna(ma200) else entry_price * 0.92

    if strategy_name == 'MACD Trend':
        return ma20 if pd.notna(ma20) else entry_price * 0.92

    return entry_price * 0.92   # fallback 8 % stop


# ── Trail / target exit condition ─────────────────────────────────────────────

def check_trail_exit(strategy_name: str, row: pd.Series) -> bool:
    """Return True when the trail or target exit is triggered on this bar."""
    close = row['Close']

    if strategy_name in ('MA Pullback', 'Trend+Pullback+Momentum'):
        ma20 = row.get('MA20', np.nan)
        return pd.notna(ma20) and close < ma20

    if strategy_name == 'Bollinger+RSI MeanReversion':
        bb_up = row.get('BB_upper', np.nan)
        return pd.notna(bb_up) and close >= bb_up   # reached target

    if strategy_name == 'Golden/Death Cross':
        ma50  = row.get('MA50',  np.nan)
        ma200 = row.get('MA200', np.nan)
        return pd.notna(ma50) and pd.notna(ma200) and ma50 < ma200

    if strategy_name == 'MACD Trend':
        dif = row.get('DIF', np.nan)
        dea = row.get('DEA', np.nan)
        return pd.notna(dif) and pd.notna(dea) and dif < dea

    return False


# ── Single-trade simulator ────────────────────────────────────────────────────

def simulate_trade(df_test: pd.DataFrame,
                   strategy_name: str,
                   entry_price: float,
                   stop_price:  float) -> dict:
    """
    Walk forward through df_test bar-by-bar from the day after the signal.
    Returns a dict with exit details and return %.
    """
    if df_test.empty:
        return None

    for i, (dt, row) in enumerate(df_test.iterrows()):
        close = float(row['Close'])
        low   = float(row['Low'])

        # Hard stop: intraday low breached the stop — exit at stop price
        if low <= stop_price:
            ep = stop_price   # assume fill near stop (could gap worse in reality)
            return dict(exit_price=ep, exit_date=dt, exit_reason='Stop Loss',
                        return_pct=(ep - entry_price) / entry_price * 100,
                        days_held=i + 1)

        # Max holding period reached
        if i + 1 >= MAX_HOLD_DAYS:
            return dict(exit_price=close, exit_date=dt, exit_reason='Max hold (1 yr)',
                        return_pct=(close - entry_price) / entry_price * 100,
                        days_held=i + 1)

        # Strategy trail / target exit
        if check_trail_exit(strategy_name, row):
            return dict(exit_price=close, exit_date=dt, exit_reason='Trail / Target',
                        return_pct=(close - entry_price) / entry_price * 100,
                        days_held=i + 1)

    # Reached end of available data before any exit triggered
    last = float(df_test['Close'].iloc[-1])
    return dict(exit_price=last, exit_date=df_test.index[-1], exit_reason='End of data',
                return_pct=(last - entry_price) / entry_price * 100,
                days_held=len(df_test))


print('Helper functions defined.')

In [ ]:
def run_backtest(stock_data: dict, separation_dates: list) -> pd.DataFrame:
    """
    For each separation date:
      - Evaluate all strategies on every ticker's training slice
      - Collect BUY signals (skip Death Cross sell warnings)
      - Simulate the trade forward; record returns
    Returns a DataFrame of all simulated trades.
    """
    records = []

    for sep_str in separation_dates:
        sep_d    = pd.Timestamp(sep_str).date()
        n_sigs   = 0

        for ticker, df in stock_data.items():
            # Date-aware split (index is UTC-aware; compare calendar dates)
            idx_dates = df.index.date
            df_train  = df[idx_dates <= sep_d]
            df_test   = df[idx_dates >  sep_d]

            if len(df_train) < 210 or len(df_test) < 5:
                continue

            # Evaluate strategies on training data only
            _, strat_results = check_all_strategies(df_train, ticker)

            # Keep only long (BUY) signals — filter out Death Cross sell warnings
            triggered = [
                r for r in strat_results
                if r['triggered']
                and not r['description'].startswith('[Death Cross')
            ]
            if not triggered:
                continue

            entry_price = float(df_train['Close'].iloc[-1])
            signal_row  = df_train.iloc[-1]
            n_strats    = len(triggered)
            investment  = n_strats * INVESTMENT_PER_STRAT
            shares      = investment / entry_price
            n_sigs     += 1

            for r in triggered:
                strat_name = r['strategy']
                stop_price = compute_stop(strat_name, entry_price, signal_row)
                result     = simulate_trade(df_test, strat_name, entry_price, stop_price)
                if result is None:
                    continue

                records.append({
                    'sep_date'    : sep_str,
                    'ticker'      : ticker,
                    'strategy'    : strat_name,
                    'n_strats'    : n_strats,
                    'investment'  : investment,
                    'entry_price' : entry_price,
                    'stop_price'  : stop_price,
                    'shares'      : shares,
                    **result,
                    'profit_usd'  : shares * (result['exit_price'] - entry_price),
                })

        print(f'  {sep_str}: {n_sigs} stocks had ≥1 BUY signal '
              f'→ {sum(1 for r in records if r["sep_date"]==sep_str)} trade records')

    return pd.DataFrame(records)


print('Backtest function defined.')

In [ ]:
print(f'Investment per strategy triggered: ${INVESTMENT_PER_STRAT:,}')
print(f'Max hold period: {MAX_HOLD_DAYS} trading days (≈ 1 year)')
print(f'Separation dates: {SEPARATION_DATES}\n')

trades = run_backtest(stock_data, SEPARATION_DATES)

print(f'\nTotal trade records: {len(trades)}')
print(f'Unique tickers:      {trades["ticker"].nunique()}')
print(f'Overall avg return:  {trades["return_pct"].mean():+.2f}%')
print(f'Overall win rate:    {(trades["return_pct"] > 0).mean()*100:.1f}%')

In [ ]:
# ── Per-strategy summary table ────────────────────────────────────────────────

summary = (
    trades.groupby('strategy')
    .agg(
        Trades        = ('return_pct', 'count'),
        Win_Rate      = ('return_pct', lambda x: f"{(x > 0).mean()*100:.1f}%"),
        Avg_Return    = ('return_pct', 'mean'),
        Median_Return = ('return_pct', 'median'),
        Best          = ('return_pct', 'max'),
        Worst         = ('return_pct', 'min'),
        Avg_Days_Held = ('days_held',  'mean'),
        Total_Profit  = ('profit_usd', 'sum'),
    )
    .round({'Avg_Return': 2, 'Median_Return': 2, 'Best': 2,
            'Worst': 2, 'Avg_Days_Held': 1, 'Total_Profit': 0})
    .sort_values('Avg_Return', ascending=False)
)

print('=== Strategy Performance Summary (all separation dates combined) ===\n')
display(
    summary.style
    .format({
        'Avg_Return'   : '{:+.2f}%',
        'Median_Return': '{:+.2f}%',
        'Best'         : '{:+.2f}%',
        'Worst'        : '{:+.2f}%',
        'Avg_Days_Held': '{:.1f}',
        'Total_Profit' : '${:,.0f}',
    })
    .background_gradient(subset=['Avg_Return'],    cmap='RdYlGn', vmin=-20, vmax=20)
    .background_gradient(subset=['Median_Return'], cmap='RdYlGn', vmin=-20, vmax=20)
    .set_caption('Higher Avg_Return = better. Best strategy is ranked first.')
)

In [ ]:
# ── Average return by separation date × strategy ──────────────────────────────

by_date = (
    trades.groupby(['sep_date', 'strategy'])['return_pct']
    .mean()
    .unstack('strategy')
    .round(2)
)

# Add row with overall mean
by_date.loc['AVERAGE'] = by_date.mean()

print('=== Average Return % by Separation Date × Strategy ===\n')
display(
    by_date.style
    .format('{:+.2f}%', na_rep='—')
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-25, vmax=25)
    .set_caption('Rows = entry date. Columns = strategy. Green = positive return.')
)

# Trade count per date
count_by_date = (
    trades.groupby(['sep_date', 'strategy'])['return_pct']
    .count()
    .unstack('strategy')
    .fillna(0).astype(int)
)
print('\n=== Trade Count per Date × Strategy ===')
display(count_by_date)

In [ ]:
# ── Exit reason breakdown ─────────────────────────────────────────────────────

exit_stats = (
    trades.groupby(['strategy', 'exit_reason'])
    .agg(
        Count      = ('return_pct', 'count'),
        Avg_Return = ('return_pct', 'mean'),
    )
    .round(2)
)

print('=== Exit Reason Breakdown (avg return & count per strategy) ===\n')
display(
    exit_stats.style
    .format({'Avg_Return': '{:+.2f}%'})
    .background_gradient(subset=['Avg_Return'], cmap='RdYlGn', vmin=-30, vmax=30)
)

# Stop loss rate per strategy
print('\n=== Stop-Loss Rate per Strategy ===')
stop_rate = (
    trades.groupby('strategy')
    .apply(lambda g: f"{(g['exit_reason']=='Stop Loss').mean()*100:.1f}%")
    .rename('Stop Loss Hit Rate')
)
display(stop_rate.to_frame())

In [ ]:
# ── Multi-strategy conviction premium ─────────────────────────────────────────
# Does investing more when multiple strategies agree pay off?

tier = (
    trades.drop_duplicates(subset=['sep_date', 'ticker', 'sep_date'])
    .groupby('n_strats')
    .agg(
        Stocks       = ('ticker',      'count'),
        Avg_Return   = ('return_pct',  'mean'),
        Win_Rate     = ('return_pct',  lambda x: f"{(x>0).mean()*100:.1f}%"),
        Avg_Invested = ('investment',  'mean'),
        Avg_Profit   = ('profit_usd',  'mean'),
    )
    .round({'Avg_Return': 2, 'Avg_Invested': 0, 'Avg_Profit': 0})
)
tier.index.name = 'N Strategies Triggered'

print('=== Investment Tiering: Does More Conviction = Better Return? ===\n')
display(
    tier.style
    .format({'Avg_Return': '{:+.2f}%', 'Avg_Invested': '${:,.0f}', 'Avg_Profit': '${:,.0f}'})
    .background_gradient(subset=['Avg_Return'], cmap='RdYlGn', vmin=-20, vmax=20)
)

In [ ]:
# ── Charts ────────────────────────────────────────────────────────────────────

from pathlib import Path
OUT_DIR = Path('test')
OUT_DIR.mkdir(exist_ok=True)

strat_order = summary.index.tolist()   # sorted by Avg_Return descending

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
fig.suptitle('AutoStock Strategy Backtest Results', fontsize=14, fontweight='bold', y=1.01)

# ── 1. Average return bar chart ───────────────────────────────────────────────
avgs   = summary['Avg_Return'].values
colors = ['#27ae60' if v >= 0 else '#c0392b' for v in avgs]
bars   = axes[0].barh(strat_order, avgs, color=colors, edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.8)
for bar, v in zip(bars, avgs):
    offset = 0.3 if v >= 0 else -0.3
    ha     = 'left' if v >= 0 else 'right'
    axes[0].text(v + offset, bar.get_y() + bar.get_height()/2,
                 f'{v:+.1f}%', va='center', ha=ha, fontsize=9, fontweight='bold')
axes[0].set_xlabel('Average Return %')
axes[0].set_title('Average Return per Strategy\n(all dates combined)')
axes[0].invert_yaxis()

# ── 2. Box plot of return distributions ──────────────────────────────────────
plot_data = [trades[trades['strategy'] == s]['return_pct'].values for s in strat_order]
bp = axes[1].boxplot(plot_data, vert=False, labels=strat_order,
                     patch_artist=True, showfliers=True,
                     flierprops=dict(marker='o', markersize=3, alpha=0.4))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('#3498db')
    patch.set_alpha(0.6)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Return %')
axes[1].set_title('Return Distribution\n(box = IQR, line = median)')

# ── 3. Win rate bar chart ─────────────────────────────────────────────────────
win_rates = [
    (trades[trades['strategy'] == s]['return_pct'] > 0).mean() * 100
    for s in strat_order
]
colors_wr = ['#27ae60' if w >= 50 else '#c0392b' for w in win_rates]
bars_wr   = axes[2].barh(strat_order, win_rates, color=colors_wr,
                          edgecolor='white', linewidth=0.5)
axes[2].axvline(50, color='black', linewidth=0.8, linestyle='--', label='50 % line')
for bar, v in zip(bars_wr, win_rates):
    axes[2].text(v + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{v:.1f}%', va='center', fontsize=9)
axes[2].set_xlabel('Win Rate %')
axes[2].set_title('Win Rate per Strategy\n(% of trades with positive return)')
axes[2].legend(fontsize=8)
axes[2].invert_yaxis()

plt.tight_layout()
out_path = OUT_DIR / 'strategy_backtest.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved → {out_path}')

In [ ]:
# ── Top & bottom trades ───────────────────────────────────────────────────────

cols_show = ['sep_date', 'ticker', 'strategy', 'entry_price',
             'stop_price', 'exit_price', 'return_pct', 'days_held', 'exit_reason']

print('=== Top 10 Best Trades ===')
display(
    trades.nlargest(10, 'return_pct')[cols_show]
    .style.format({'entry_price': '${:.2f}', 'stop_price': '${:.2f}',
                   'exit_price': '${:.2f}', 'return_pct': '{:+.1f}%'})
    .background_gradient(subset=['return_pct'], cmap='Greens')
)

print('\n=== Top 10 Worst Trades ===')
display(
    trades.nsmallest(10, 'return_pct')[cols_show]
    .style.format({'entry_price': '${:.2f}', 'stop_price': '${:.2f}',
                   'exit_price': '${:.2f}', 'return_pct': '{:+.1f}%'})
    .background_gradient(subset=['return_pct'], cmap='Reds_r')
)

In [ ]:
# ── Final verdict ─────────────────────────────────────────────────────────────

print('=' * 60)
print('FINAL VERDICT — Average Return by Strategy')
print('=' * 60)
for strat in strat_order:
    sub    = trades[trades['strategy'] == strat]
    avg_r  = sub['return_pct'].mean()
    wins   = (sub['return_pct'] > 0).mean() * 100
    n      = len(sub)
    bar    = '█' * int(max(avg_r, 0) / 2) + '░' * int(max(-avg_r, 0) / 2)
    sign   = '▲' if avg_r > 0 else '▼'
    print(f'  {sign} {strat:<35s}  {avg_r:+6.2f}%  win={wins:.0f}%  n={n}  {bar}')

best = summary.index[0]
print(f'\n  Best strategy overall: {best}')
print(f'  Avg return: {summary.loc[best, "Avg_Return"]:+.2f}%')
print(f'  Win rate  : {summary.loc[best, "Win_Rate"]}')